# ImmigrationNavigator — RAG Pipeline

---

## Goal
Build a Retrieval-Augmented Generation (RAG) pipeline over the USCIS corpus collected in NB1.
Every answer is grounded in official USCIS sources and includes a citation.

## Pipeline
```
S3 (raw_docs.json)
      ↓
Text cleaning + chunking
      ↓
FastEmbed embeddings → ChromaDB vector store
      ↓
User question → retrieve top-k chunks
      ↓
Groq LLM (Llama 3.1) → cited answer
```

## MVP Questions
1. When do I need to apply for OPT, and what forms do I need?
2. Am I eligible for a STEM OPT extension, and what are the deadlines?
3. What happens to my status during the H-1B cap-gap period?

## Cell 1 — Install Dependencies

In [ ]:
!pip install langchain langchain-groq langchain-community \
             chromadb fastembed tiktoken boto3 -q

## Cell 2 — Imports and Setup

Loads the Groq API key from AWS Secrets Manager and initializes all libraries.

In [ ]:
import os
import re
import json
import uuid
import boto3
from datetime import datetime

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate
from fastembed import TextEmbedding
import chromadb

# Load Groq API key from AWS Secrets Manager
def get_secret(secret_name):
    client = boto3.client("secretsmanager", region_name="us-east-1")
    response = client.get_secret_value(SecretId=secret_name)
    return json.loads(response["SecretString"])

secrets = get_secret("immigration-navigator/groq")
os.environ["GROQ_API_KEY"] = secrets["GROQ_API_KEY"]

# S3 client
s3 = boto3.client("s3", region_name="us-east-1")
S3_BUCKET = "immigration-navigator-data"

print("✅ Setup complete")

## Cell 3 — Load Corpus from S3

Loads `raw_docs.json` from S3 and filters documents with enough content for RAG (>200 words).

> **Note:** `f1_chapter5` is renamed to `opt_and_stem_opt` because it contains the actual OPT/STEM OPT policy text.
> The `opt_overview` / `opt_chapter*` URLs point to Adjustment of Status (wrong content) — those are relabeled accordingly.

In [ ]:
# Load from S3
response = s3.get_object(Bucket=S3_BUCKET, Key="raw_docs.json")
all_docs = json.loads(response["Body"].read().decode("utf-8"))

# Filter short documents
good_docs = [d for d in all_docs if d["word_count"] > 200]

# Fix labels — Volume 7 Part A is Adjustment of Status, not OPT
# Real OPT content is in Volume 2 Part F Chapter 5
label_fixes = {
    "opt_overview": "adj_status_overview",
    "opt_chapter1": "adj_status_ch1",
    "opt_chapter2": "adj_status_ch2",
    "opt_chapter3": "adj_status_ch3",
    "opt_chapter4": "adj_status_ch4",
    "opt_chapter5": "adj_status_ch5",
    "f1_chapter5":  "opt_and_stem_opt",
}
for d in good_docs:
    if d["label"] in label_fixes:
        d["label"] = label_fixes[d["label"]]

print(f"Total docs loaded : {len(all_docs)}")
print(f"Docs ready for RAG: {len(good_docs)}")
print(f"Total words       : {sum(d['word_count'] for d in good_docs):,}")

## Cell 4 — Text Cleaning and Chunking

USCIS pages contain navigation menus and boilerplate mixed with policy content.
We strip those before chunking to avoid polluting the vector store with useless text.

- Chunk size: 800 tokens
- Overlap: 150 tokens (preserves context at boundaries)
- Min chunk size: 50 words (filters nav fragments)

In [ ]:
def clean_text(text):
    """
    Remove USCIS navigation boilerplate from scraped HTML text.
    Strips nav menus, CFR/INA inline references, volume listings,
    and browser security banners before chunking.
    """
    patterns = [
        r'Policy Manual\s*\n.*?Feedback',           # Nav header
        r'USCIS-PM\s*\n.*?Volume \d+.*?\n',          # Volume references
        r'Affected Sections.*?Volume \d+.*?\n',       # Affected sections
        r'Skip to main content.*?secure websites\.',  # Browser boilerplate
        r'Countdown to America.*?Minutes',            # Anniversary banner
        r'An official website.*?HTTPS',               # Gov website banner
        r'\d+\s*USCIS-PM\s*-\s*\n',                  # Volume numbers
        r'Contents\s*\nUpdates\s*\nINA\s*\n8 CFR',   # Nav menu
        r'8 CFR \d+\.\d+.*?\n',                       # CFR references inline
        r'INA \d+.*?\n',                              # INA references inline
    ]
    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.DOTALL | re.IGNORECASE)

    # Remove lines shorter than 4 words (usually nav items)
    lines = [l for l in text.split('\n') if len(l.split()) >= 4]
    text = '\n'.join(lines)

    # Normalize whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " "]
)

# Build LangChain Document objects with cleaned text
lc_docs = []
for d in good_docs:
    clean = clean_text(d["text"])
    if len(clean.split()) > 100:
        lc_docs.append(Document(
            page_content=clean,
            metadata={
                "source": d["source"],
                "label":  d["label"],
                "url":    d.get("url", ""),
            }
        ))

# Chunk and filter very short chunks
chunks = splitter.split_documents(lc_docs)
chunks = [c for c in chunks if len(c.page_content.split()) > 50]

print(f"Documents : {len(lc_docs)}")
print(f"Chunks    : {len(chunks)}")
print(f"Avg size  : {sum(len(c.page_content.split()) for c in chunks) // len(chunks)} words")

## Cell 5 — Embeddings and Vector Store

Generates embeddings with FastEmbed (`BAAI/bge-small-en-v1.5`) and stores them in ChromaDB.

- ChromaDB is persisted to `/home/sagemaker-user/chroma_db` so it survives session restarts
- `get_or_create_collection` means re-running this cell won't duplicate chunks
- First run downloads the embedding model (~80MB) — takes 1-2 minutes

In [ ]:
# Load embedding model
embedding_model = TextEmbedding("BAAI/bge-small-en-v1.5")
print("✅ Embedding model loaded")

# Initialize persistent ChromaDB
chroma_client = chromadb.PersistentClient(path="/home/sagemaker-user/chroma_db")
collection = chroma_client.get_or_create_collection(
    name="immigration_navigator",
    metadata={"hnsw:space": "cosine"}
)

# Only insert if collection is empty — avoids duplicates on re-run
if collection.count() == 0:
    BATCH_SIZE = 50
    documents = [c.page_content for c in chunks]
    metadatas = [c.metadata for c in chunks]
    ids = [str(uuid.uuid4()) for _ in chunks]

    print(f"Inserting {len(chunks)} chunks in batches of {BATCH_SIZE}...")
    for i in range(0, len(chunks), BATCH_SIZE):
        batch_docs  = documents[i:i+BATCH_SIZE]
        batch_meta  = metadatas[i:i+BATCH_SIZE]
        batch_ids   = ids[i:i+BATCH_SIZE]
        embeddings  = [e.tolist() for e in embedding_model.embed(batch_docs)]
        collection.add(
            documents=batch_docs,
            embeddings=embeddings,
            metadatas=batch_meta,
            ids=batch_ids
        )
        if (i // BATCH_SIZE) % 5 == 0:
            print(f"  {min(i+BATCH_SIZE, len(chunks))}/{len(chunks)} chunks")
else:
    print(f"Collection already has {collection.count()} chunks — skipping insert")

print(f"\n✅ ChromaDB ready — {collection.count()} chunks")

## Cell 6 — LLM Setup

Initializes Groq LLM with a citation-enforced system prompt.
The prompt instructs the model to:
- Answer ONLY from retrieved context
- Cite every claim with `[Source: ...]`
- Decline to answer if context is insufficient

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=secrets["GROQ_API_KEY"],
    temperature=0
)

PROMPT = ChatPromptTemplate.from_template("""
You are ImmigrationNavigator, an AI assistant that helps international students
navigate U.S. visa processes. Answer ONLY using the provided context.
For every claim you make, cite the source in brackets like [Source: USCIS Policy Manual, opt_and_stem_opt].
If the context does not contain enough information to answer, say:
"I don't have enough information to answer this question. Please consult your ISO or an immigration attorney."

Context:
{context}

Question: {question}

Answer (cite every claim):
""")

print("✅ LLM initialized — llama-3.1-8b-instant via Groq")

## Cell 7 — RAG Pipeline

The `ask()` function is the full pipeline:
1. Embed the user question
2. Retrieve top-k most relevant chunks from ChromaDB
3. Build a context string with source metadata
4. Pass context + question to the LLM
5. Return a cited answer

In [ ]:
def ask(question, n_results=5):
    """
    Full RAG pipeline: retrieve relevant USCIS chunks and generate a cited answer.

    Args:
        question  (str): User's natural language question.
        n_results (int): Number of chunks to retrieve (default 5).

    Returns:
        str: LLM-generated answer with inline citations.
    """
    # Step 1 — Embed the question
    query_embedding = list(embedding_model.embed([question]))[0].tolist()

    # Step 2 — Retrieve top-k chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas"]
    )

    # Step 3 — Build context with source metadata
    context_parts = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        context_parts.append(
            f"[Source: {meta['source']}, {meta['label']}, {meta['url']}]\n{doc}"
        )
    context = "\n\n".join(context_parts)

    # Step 4 — Generate cited answer
    chain = PROMPT | llm
    response = chain.invoke({"context": context, "question": question})
    return response.content

## Cell 8 — Test: 3 MVP Questions

Validates the pipeline against the three core questions the MVP must answer.

In [ ]:
questions = [
    "When do I need to apply for OPT and what forms do I need?",
    "Am I eligible for a STEM OPT extension and what are the deadlines?",
    "What happens to my status during the H-1B cap-gap period?",
]

for q in questions:
    print(f"{'='*60}")
    print(f"Q: {q}\n")
    print(ask(q))
    print()

## Cell 9 — Save Pipeline State to S3

Saves metadata about the current pipeline configuration to S3
so the team can track versioning and reproduce results.

In [ ]:
pipeline_state = {
    "embedding_model": "BAAI/bge-small-en-v1.5",
    "llm_model":       "llama-3.1-8b-instant",
    "llm_provider":    "Groq",
    "chunks_count":    collection.count(),
    "chunk_size":      800,
    "chunk_overlap":   150,
    "min_chunk_words": 50,
    "saved_at":        datetime.now().isoformat()
}

s3.put_object(
    Bucket=S3_BUCKET,
    Key="pipeline_state.json",
    Body=json.dumps(pipeline_state, indent=2).encode("utf-8")
)

print("✅ Pipeline state saved to S3")
print(json.dumps(pipeline_state, indent=2))